### Similarities berechnen, nachdem die Thesen klassifiziert wurden
#### (Spielereien)
---

In [49]:
# pip install pyarrow

In [43]:
# pip install sentence_transformers

In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [3]:
# Dateien einlesen
df_modified = pd.read_parquet("all_parties_text_combined_prep_classified.parquet")
df_wahlomat = pd.read_parquet("wahlomaten_thesen_positionen_classified.parquet")

distilbert nur für englische Texte geeignet

In [ ]:
# Lade das Sentence-Transformer-Modell für die Ähnlichkeitsberechnung
model = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')

# DataFrame für die Ergebnisse initialisieren
results = []

# Iteriere über die These im DataFrame df_wahlomat
for index, row in df_wahlomat.iterrows():
    Predicted_Class = row['Predicted_Class']
    wahlomat_these = row['These']
    
    # Filtere die Zeilen im df_modified mit derselben Predicted_Class
    matching_rows = df_modified[df_modified['Predicted_Class'] == Predicted_Class]
    
    # Berechne die Ähnlichkeiten
    for _, match_row in matching_rows.iterrows():
        # Texte aus den Spalten "These" und "Text" kombinieren
        text_to_compare = match_row['These'] + " " + match_row['Text']
        
        # Berechne die Ähnlichkeiten
        embeddings = model.encode([wahlomat_these, text_to_compare])
        similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

        # Ergebnisse speichern
        results.append({
            'These': wahlomat_these,
            'Party': match_row['Party'],
            'Similarity': similarity
        })

# Ergebnisse in ein DataFrame umwandeln
similarity_df = pd.DataFrame(results)

# Optional: Ergebnisse nach Ähnlichkeit sortieren
similarity_df = similarity_df.sort_values(by='Similarity', ascending=False)

# Ausgabe der Ergebnisse
print(similarity_df)


                                                 These  Party  Similarity
151  Asylsuchende, die über einen anderen EU-Staat ...    fdp    0.859671
143  Asylsuchende, die über einen anderen EU-Staat ...    afd    0.851794
145  Asylsuchende, die über einen anderen EU-Staat ...    afd    0.847679
142  Asylsuchende, die über einen anderen EU-Staat ...    afd    0.845624
605  Die Ausbildungsförderung BAföG soll weiterhin ...  linke    0.837842
..                                                 ...    ...         ...
89   Alle Bürgerinnen und Bürger sollen in gesetzli...  linke    0.461690
188  Auf allen Autobahnen soll ein generelles Tempo...    fdp    0.434017
164  Auf allen Autobahnen soll ein generelles Tempo...    cdu    0.432489
221  Aus Deutschland sollen weiterhin Rüstungsgüter...    fdp    0.419510
187  Auf allen Autobahnen soll ein generelles Tempo...    fdp    0.388221

[960 rows x 3 columns]


In [ ]:
# similarity_df.to_csv("test.csv", index=False, encoding="utf-8-sig")

MiniLM auch für andere Sprachen geeignet

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Lade das Modell für die Ähnlichkeitsberechnung
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Ähnlichkeitsabgleich für die Thesen im df_wahlomat
results = []

for index, row in df_wahlomat.iterrows():
    wahlomat_these = row['These']
    Predicted_Class = row['Predicted_Class']
    
    # Filtere df_modified nach Predicted_Class
    filtered_df = df_modified[df_modified['Predicted_Class'] == Predicted_Class]

    for _, modified_row in filtered_df.iterrows():
        modified_these = modified_row['These'] + ": " + modified_row['Text']  # Kombination von These und Text
        # modified_these = modified_row['These']
        party = modified_row['Party']

        # Berechne die Ähnlichkeit zwischen den Thesen
        embeddings = model.encode([wahlomat_these, modified_these])
        similarity = util.pytorch_cos_sim(embeddings[0], embeddings[1]).item()

        results.append({
            'These_Wahlomat': wahlomat_these,
            'These_Modified': modified_these,
            'Party': party,
            'Similarity': similarity
        })

# Erstelle ein DataFrame aus den Ergebnissen
results_df = pd.DataFrame(results)

# Optional: Ergebnisse nach Ähnlichkeit sortieren
results_df = results_df.sort_values(by='Similarity', ascending=False)

# Ausgabe der Ergebnisse
print(results_df)


                                        These_Wahlomat  \
951  Ökologische Landwirtschaft soll stärker geförd...   
953  Ökologische Landwirtschaft soll stärker geförd...   
419  Der Bund soll Projekte gegen Rechtsextremismus...   
316  Beim Ausbau der Verkehrsinfrastruktur soll die...   
509  Deutschland soll weiterhin die Anwerbung von F...   
..                                                 ...   
269  Bei der Besteuerung von Einkommen soll der Spi...   
30   Alle Beschäftigten sollen bereits nach 40 Beit...   
132  An Bahnhöfen soll die Bundespolizei Software z...   
616  Die Ausbildungsförderung BAföG soll weiterhin ...   
120  An Bahnhöfen soll die Bundespolizei Software z...   

                                        These_Modified   Party  Similarity  
951           Stärkung der ökologischen Landwirtschaft  gruene    0.829187  
953                  Bäuerliche Landwirtschaft stärken     afd    0.745493  
419    Wir wollen Extremismus vorbeugen und bekämpfen.     spd    0.7376

In [ ]:
# Behalte pro These_Wahlomat und Party nur die Zeile mit der höchsten Similarity
results_df = results_df.loc[results_df.groupby(['These_Wahlomat', 'Party'])['Similarity'].idxmax()]

# results_df.to_csv("test.csv", index=False, encoding="utf-8-sig")

results_df.head()

,These_Wahlomat,These_Modified,Party,Similarity
26,Alle Beschäftigten sollen bereits nach 40 Beit...,"Grundsicherung im Alter: Wer gearbeitet hat, m...",afd,0.451782
9,Alle Beschäftigten sollen bereits nach 40 Beit...,Rente und Altersvorsorge langfristig sichern,cdu,0.548687
31,Alle Beschäftigten sollen bereits nach 40 Beit...,"Für flexible, finanzierbare und steigende Renten",fdp,0.403819
23,Alle Beschäftigten sollen bereits nach 40 Beit...,Schutz vor Altersarmut durch gute Renten,gruene,0.552331
43,Alle Beschäftigten sollen bereits nach 40 Beit...,Eine sichere Rente für alle,linke,0.546511


In [71]:
# Pivot-Tabelle erstellen
pivot_table = results_df.pivot_table(
    index='These_Wahlomat',         # Zeilenüberschrift
    columns='Party',                # Spaltenüberschrift
    values='Similarity',            # Werte in der Tabelle
    aggfunc='mean'                  # Aggregationsfunktion (z.B. Durchschnitt)
)

# Ausgabe der Pivot-Tabelle
pivot_table.head()

Party,afd,cdu,fdp,gruene,linke,spd
These_Wahlomat,,,,,,
Alle Beschäftigten sollen bereits nach 40 Beitragsjahren ohne Abschläge in Rente gehen können.,0.451782,0.548687,0.403819,0.552331,0.546511,0.527030
Alle Bürgerinnen und Bürger sollen in gesetzlichen Krankenkassen versichert sein müssen.,0.431225,0.467642,0.546057,0.520831,0.431866,0.522472
An Bahnhöfen soll die Bundespolizei Software zur automatisierten Gesichtserkennung einsetzen dürfen.,0.338926,0.419406,0.258161,0.365309,0.051955,0.289132
Asylsuchende sollen in Deutschland sofort nach ihrer Antragstellung eine Arbeitserlaubnis erhalten.,NaN,NaN,0.395606,0.294464,NaN,NaN
"Asylsuchende, die über einen anderen EU-Staat eingereist sind, sollen an den deutschen Grenzen abgewiesen werden.",0.491953,0.448502,0.386446,NaN,NaN,0.307796
